In [ ]:
import matplotlib.pyplot as plt
plt.style.use("utils/RESS.mplstyle")

## Figure 13

Scalability study

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

ccycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

comps = ["classical_cuda_float", "matheron_cuda_float"]
labels = ["Classical", "Pathwise"]
lstyles = ["-", "-"]
ccolors = [ccycle[0], ccycle[1]]

output_folder = "_experiments/RESS/scalability"

fig, ax = plt.subplots(1,2, figsize=np.array([230, 100]) / 72.27)

# ---------------------------------------------------------------------
# Subplot 1: Sampling time
# ---------------------------------------------------------------------
sample_time = []
t_estimation = []

for idx, comp in enumerate(comps):
    filename = f"{output_folder}/{comp}.npz"
    data = np.load(filename)

    q_sample_vec = data["J_sample_vec"]
    t_vari_n_q = data["t_vari_n_q"]

    t_estimation.append(t_vari_n_q[-1])
    sample_time.append(t_vari_n_q[-1])

    ax[0].plot(
        q_sample_vec,
        t_vari_n_q,
        linestyle=lstyles[idx],
        label=labels[idx],
        color=ccolors[idx],
        linewidth = 1.5,
    )

ax[0].set_xlabel("Sample size")
ax[0].set_yticks([0, 0.5, 1.0])
ax[0].set_xticks([0, 5000, 10000])
ax[0].grid(axis="y", linestyle="--", alpha=0.5)
ax[0].set_ylabel("Time (s)")
# ax[0].legend(frameon=False)
sns.despine(ax=ax[0])

# ---------------------------------------------------------------------
# Calculate optimization timings
# ---------------------------------------------------------------------
t_estimation = np.asarray(t_estimation).reshape(1, -1)

n_evals = np.array(
    [
        51 * 15 * 1,
        51 * 15 * 5,
        51 * 15 * 10,
    ]
).reshape(-1, 1)

# Shape: (3 dimensions, 2 methods)
opt_times = n_evals @ t_estimation

# ---------------------------------------------------------------------
# Subplot 2: Optimization timing comparison
# ---------------------------------------------------------------------
dimensions = ["1d", "5d", "10d"]

classical_timings = opt_times[:, 0]/60
pathwise_timings = opt_times[:, 1]/60

x = np.arange(len(dimensions))

width = 0.30
spacing = 0.05
offset = (width + spacing) / 2

classical_bars = ax[1].bar(
    x - offset,
    classical_timings,
    width=width,
    label="Classical",
    color=ccycle[0],
)

pathwise_bars = ax[1].bar(
    x + offset,
    pathwise_timings,
    width=width,
    label="Pathwise",
    color=ccycle[1],
)

# Add values above the bars
ax[1].bar_label(
    classical_bars,
    labels=[f"{value:.0f}" for value in classical_timings],
    padding=3,
    fontsize=6,
)

pathwise_labels = ax[1].bar_label(
    pathwise_bars,
    labels=[f"{value:.1f}" for value in pathwise_timings],
    padding=3,
    fontsize=6,
)

# Shift labels slightly to the right
for label in pathwise_labels:
    label.set_position((2, 1))  # 5 points right, 3 points above

ax[1].set_xlabel("Problem dimension")
ax[1].set_ylabel("Time (min)")
ax[1].set_xticks(x)
ax[1].set_xticklabels(dimensions)

ax[1].grid(axis="y", linestyle="--", alpha=0.5)
ax[1].set_axisbelow(True)
sns.despine(ax=ax[1])

# Leave space for the labels above the bars
all_timings = np.concatenate([classical_timings, pathwise_timings])
ax[1].set_ylim(top=all_timings.max() * 1.18)

# Figure-level legend containing only bar-plot entries
fig.legend(
    handles=[classical_bars, pathwise_bars],
    labels=["Classical", "Matheron"],
    loc="upper center",
    bbox_to_anchor=(0.5, 1.05),
    ncol=2,
    frameon=False,
)

fig.tight_layout(rect=[0, 0, 1, 0.92])

# Save the combined figure
plot_folder = os.path.join("_plots", "RESS")
os.makedirs(plot_folder, exist_ok=True)



fig.text(
    0.035, 0.08, r"\textsf{\textbf{(a)}}",
    ha="left",
    va="bottom",
    fontsize=8,
    color="black",
)

fig.text(
    0.525, 0.08, r"\textsf{\textbf{(b)}}",
    ha="left",
    va="bottom",
    fontsize=8,
    color="black",
)

file_name = "scalability.pdf"
file_path = os.path.join(plot_folder, file_name)

fig.savefig(
    file_path,
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.1,
)
